In [1]:
# @title Load Human Data

from google.colab import drive
drive.mount('/content/drive')

dataset_dir = "/content/drive/MyDrive/LLM Auction-release/survey/"

import pandas as pd

df = pd.read_csv(dataset_dir + 'survey_data_anonymized.csv')

print(df.shape)

Mounted at /content/drive
(48, 17)


In [2]:
def col_summary(df, key_column):
    counts = df[key_column].value_counts(dropna=False)
    percent = df[key_column].value_counts(normalize=True, dropna=False) * 100
    print(percent)
    print()

col_summary(df, "6.Education Level")
col_summary(df, "8.English Language Level")
col_summary(df, "9.How frequently do you see advertisements while browsing online?")
col_summary(df, "11.On a scale of 1 to 5， how acceptable do you find advertisements embedded within responses from AI chatbots (e.g.， ChatGPT， Google Gemini)?")
col_summary(df, "13.When do you expect advertisements will start appearing in the responses of AI chatbots like ChatGPT?")

6.Education Level
D.Master's Degree      50.000000
C.Bachelor's Degree    31.250000
E.Doctoral Degree      16.666667
B.Associate Degree      2.083333
Name: proportion, dtype: float64

8.English Language Level
C.Advanced           43.750000
D.Fluent             25.000000
B.Intermediate       22.916667
E.Second-language     8.333333
Name: proportion, dtype: float64

9.How frequently do you see advertisements while browsing online?
A.Very frequently – I see ads on almost every website or app I use    45.833333
B.Frequently – I notice ads several times a day                       35.416667
C.Occasionally – I see ads a few times a week                         16.666667
D.Rarely – I only see ads once in a while                              2.083333
Name: proportion, dtype: float64

11.On a scale of 1 to 5， how acceptable do you find advertisements embedded within responses from AI chatbots (e.g.， ChatGPT， Google Gemini)?
B.2 – Slightly acceptable      45.833333
C.3 – Moderately acceptable   

In [3]:
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np
from textblob import TextBlob
import jieba
from wordcloud import WordCloud

# Survey responses data
responses = df["12.Do you have any concerns about advertisements being included in AI chatbot responses? If so， please describe them.  (You may use any language， though English is encouraged)"]

class SurveyAnalyzer:
    def __init__(self, responses):
        self.responses = responses
        self.total_responses = len(responses)

    def categorize_responses(self):
        """Categorize responses into concern types"""
        categories = {
            'no_concerns': 0,
            'trust_objectivity': 0,
            'difficulty_distinguishing': 0,
            'misinformation': 0,
            'user_experience': 0,
            'privacy': 0,
            'conditional_acceptance': 0,
            'quality_concerns': 0
        }

        # Keywords for each category
        trust_objectivity_keywords = ['trust', 'objectivity', 'objective', 'bias', 'neutral', 'credibility', 'confusion',
                         'reliability', 'suspicious', 'distinguish', 'identify', 'hidden', 'subtle', 'unconscious',
                          '客观', '信任', '可信', '偏见', '中性', '区分', '分辨', '隐藏', '无意识']

        misinfo_keywords = ['misinformation', 'false', 'misleading', 'accuracy', 'incorrect',
                           '虚假', '误导', '准确', '错误', '幻觉']

        ux_keywords = ['experience', 'interrupt', 'distract', 'flow', 'intrusive', 'efficiency',
                       'unrelated', '体验', '打断', '干扰', '效率']

        privacy_keywords = ['privacy', 'personal', 'targeted', '隐私', '个人', '针对性']

        quality_keywords = ['quality', 'lower', 'degrade', 'best answer', '质量', '降低']

        for response in self.responses:
            response_lower = response.lower()

            # Filter out too short responses
            if len(response.strip()) < 10:
                continue
            else:
                # Check each categories
                if any(keyword in response_lower for keyword in trust_objectivity_keywords):
                    categories['trust_objectivity'] += 1
                if any(keyword in response_lower for keyword in misinfo_keywords):
                    categories['misinformation'] += 1
                if any(keyword in response_lower for keyword in ux_keywords):
                    categories['user_experience'] += 1
                if any(keyword in response_lower for keyword in privacy_keywords):
                    categories['privacy'] += 1
                if any(keyword in response_lower for keyword in quality_keywords):
                    categories['quality_concerns'] += 1

        return categories

    def calculate_response_lengths(self):
        """Analyze response lengths"""
        lengths = [len(response.strip()) for response in self.responses if response.strip()]
        return {
            'mean_length': np.mean(lengths),
            'median_length': np.median(lengths),
            'std_length': np.std(lengths),
            'min_length': min(lengths),
            'max_length': max(lengths)
        }

analyzer = SurveyAnalyzer(responses)

print("QUANTITATIVE ANALYSIS OF AI ADVERTISEMENT SURVEY")

print(f"Total responses: {analyzer.total_responses}")

length_stats = analyzer.calculate_response_lengths()
print(f"\nRESPONSE LENGTH STATISTICS")
print(f"Mean length: {length_stats['mean_length']:.1f} characters")
print(f"Median length: {length_stats['median_length']:.1f} characters")
print(f"Standard deviation: {length_stats['std_length']:.1f} characters")
print(f"Range: {length_stats['min_length']} - {length_stats['max_length']} characters")

categories = analyzer.categorize_responses()
print(f"\nCONCERN CATEGORIES")
print(f"Note: Responses can belong to multiple categories")
print()
for category, count in sorted(categories.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / analyzer.total_responses) * 100
    category_name = category.replace('_', ' ').title()
    print(f"{category_name}: {count} ({percentage:.1f}%)")

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


QUANTITATIVE ANALYSIS OF AI ADVERTISEMENT SURVEY
Total responses: 48

RESPONSE LENGTH STATISTICS
Mean length: 91.1 characters
Median length: 61.5 characters
Standard deviation: 90.8 characters
Range: 2 - 358 characters

CONCERN CATEGORIES
Note: Responses can belong to multiple categories

Trust Objectivity: 18 (37.5%)
Misinformation: 10 (20.8%)
User Experience: 6 (12.5%)
Quality Concerns: 4 (8.3%)
Privacy: 1 (2.1%)
No Concerns: 0 (0.0%)
Difficulty Distinguishing: 0 (0.0%)
Conditional Acceptance: 0 (0.0%)
